# Module 12: Display It!

## Mission Briefing

So far, every time you wanted to see what your Pico was thinking, you had to look at your computer screen using `print()`. But what happens when you unplug the USB cable? All those readings... gone.

Today you give your Pico its **own screen**. A tiny 0.96-inch OLED display that sits right on your breadboard. Text, numbers, sensor readings — all visible without a computer.

```
                           +--------------------+
                           |      Pico 2W       |
    +----------+  I2C     |                    |  I2C
    |  0.96"   | <------->| GP0 (SDA)          |
    |   OLED   |  2 wires |                    |
    | Display  | <------->| GP1 (SCL)          |
    +----------+          +--------------------+
```

Your Pico just got a face!

---
## Science Spotlight: I2C Communication

So far we've been using `print()` to see data — but that only works when the Pico is connected to a computer. What if we had a tiny screen right on the breadboard?

The OLED display talks to the Pico using something called **I2C** — a special communication protocol that uses just **2 wires** to send data back and forth. Each device on the I2C bus has its own **address**, like a house number on a street.

*Your instructor will explain how I2C works and why only 2 wires can connect to many different devices.*

---
## Meet the OLED Display

Your OLED display is a **128 x 64 pixel** screen. That means 128 dots across and 64 dots down. It's tiny, but it can show text, numbers, and even simple drawings.

The display module has **4 pins**:

```
     +-------------------+
     |                   |
     |   OLED Display    |
     |     128 x 64      |
     |                   |
     +---+---+---+---+---+
         |   |   |   |
        VCC GND SDA SCL
         |   |   |   |
       3.3V GND GP0 GP1
```

- **VCC** = power (3.3V)
- **GND** = ground
- **SDA** = data line (sends information back and forth)
- **SCL** = clock line (keeps the timing synchronized)

**Important:** OLED stands for **Organic Light-Emitting Diode**. Each pixel makes its own light — no backlight needed. That's why black pixels are truly OFF (saving power).

---
## Wiring: OLED Display

Only 4 wires needed!

### Circuit

```
   Pico 3.3V (pin 36) ──── wire ──── OLED VCC
   Pico GND  (pin 38) ──── wire ──── OLED GND
   Pico GP0  (pin 1)  ──── wire ──── OLED SDA
   Pico GP1  (pin 2)  ──── wire ──── OLED SCL
```

### Step-by-Step

1. **Place the OLED display** on the breadboard — insert the 4 header pins into 4 separate rows
2. **Connect a wire** from **3.3V** on the Pico to the **VCC** pin of the OLED
3. **Connect a wire** from **GND** on the Pico to the **GND** pin of the OLED
4. **Connect a wire** from **GP0** on the Pico to the **SDA** pin of the OLED
5. **Connect a wire** from **GP1** on the Pico to the **SCL** pin of the OLED

**Double-check:** Make sure VCC goes to 3.3V, NOT 5V! The OLED display runs on 3.3V.

Plug in your USB cable.

---
## Important: Upload the ssd1306 Library

The OLED display uses a chip called **SSD1306**. MicroPython doesn't have built-in support for it, so we need to upload a special **library file** to the Pico.

**Ask your instructor to help you upload `ssd1306.py` to your Pico before continuing.**

Once the file is on your Pico, you can use it in your code with `import ssd1306`.

---
## Code Along: Scan for I2C Devices

Before we try to use the display, let's make sure the Pico can actually **find** it on the I2C bus. Every I2C device has an address — the OLED is usually at address `0x3C` (which is 60 in decimal).

Fill in the blanks:

In [ ]:
from machine import Pin, I2C

i2c = I2C(0, sda=Pin(_____), scl=Pin(_____))    # Which GPIO pins did we wire SDA and SCL to?

devices = i2c.scan()
print("I2C devices found:", devices)

if devices:
    for d in devices:
        print("  Address:", d, "(hex: " + hex(d) + ")")
else:
    print("No devices found! Check your wiring.")

You should see `[60]` in the output — that's your OLED display at address `0x3C` (60 in decimal).

If you see an empty list `[]`, double-check your wiring — especially SDA and SCL.

---
## Code Along: Hello, OLED!

Now let's put some text on the screen! We use the `ssd1306` library to control the display.

The display works like a canvas:
1. You **draw** things on it (text, shapes)
2. Nothing appears until you call `oled.show()`

Fill in the blanks:

In [ ]:
from machine import Pin, I2C
from ssd1306 import SSD1306_I2C

i2c = I2C(0, sda=Pin(0), scl=Pin(1))
oled = SSD1306_I2C(_____, _____, i2c)    # Width and height of the display in pixels?

oled.text("Hello!", 0, 0)                # text, x position, y position
oled._____()                              # What command sends the drawing to the screen?

You should see "Hello!" in the top-left corner of the display!

The `oled.text()` command takes 3 arguments:
- The text to display (in quotes)
- **x** position (0 = left edge, goes up to 127)
- **y** position (0 = top edge, goes up to 63)

Each character is 8 pixels wide and 8 pixels tall.

---
## Code Along: Multiple Lines of Text

Since each character is 8 pixels tall, you can fit about 8 lines of text on the 64-pixel-tall screen. Let's display multiple lines.

Fill in the blanks:

In [ ]:
from machine import Pin, I2C
from ssd1306 import SSD1306_I2C

i2c = I2C(0, sda=Pin(0), scl=Pin(1))
oled = SSD1306_I2C(128, 64, i2c)

oled.fill(0)                          # Clear the screen (fill with black)
oled.text("Circuit", 0, _____)        # Line 1 — what y position for the top?
oled.text("Explorers!", 0, _____)     # Line 2 — 8 pixels below line 1
oled.text("Module 12", 0, _____)      # Line 3 — 8 pixels below line 2
oled.show()

**Pattern:** Each line is 8 pixels apart. Line 1 starts at y=0, line 2 at y=8, line 3 at y=16, and so on.

Try changing the text to your own messages!

---
## Code Along: Live Counter

Now let's make the display **update** — showing a counter that goes up every second. This is how you'd display changing sensor data.

The key: you must **clear** the screen before drawing new content, or old text will overlap.

Fill in the blanks:

In [ ]:
from machine import Pin, I2C
from ssd1306 import SSD1306_I2C
import time

i2c = I2C(0, sda=Pin(0), scl=Pin(1))
oled = SSD1306_I2C(128, 64, i2c)

count = 0

while True:
    oled._____(0)                     # Clear the screen first!
    oled.text("Counter:", 0, 0)
    oled.text(str(_____), 0, 16)      # What variable holds our count? (Hint: must convert to string)
    oled._____()                      # Send to screen
    count = count + _____             # Increase the counter by 1
    time.sleep(1)

You should see the counter go up every second on the OLED screen!

**Key pattern for updating displays:**
1. `oled.fill(0)` — clear the screen
2. `oled.text(...)` — draw new content
3. `oled.show()` — send it to the display
4. Repeat!

---
## Experiments

Try these modifications:

1. **Move it around:** Change the x and y positions in `oled.text()` to put text in different places on the screen. Can you put text in the center? (Hint: the screen is 128 wide, each character is 8 pixels wide, so "Hello" which is 5 characters = 40 pixels. Center would be x = (128 - 40) / 2 = 44)

2. **Invert the display:** Try `oled.invert(True)` — what happens? Use `oled.invert(False)` to go back.

3. **Draw a rectangle:** The display can also draw shapes! Try:
   ```python
   oled.rect(0, 0, 128, 64, 1)    # x, y, width, height, color (1=white)
   oled.show()
   ```

4. **Countdown:** Modify the counter code to count DOWN from 10 to 0, then display "BLAST OFF!" at the end.

---
## Challenge: Digital Name Badge

Create a **digital name badge** on the OLED display!

### Requirements:
1. Display your **name** in large-ish text
2. Display a **fun fact** about yourself
3. Add an **animation** — make the text blink on and off, or scroll between two screens

### Useful hints:

```python
# Blinking text effect:
while True:
    oled.fill(0)
    oled.text("Your Name", 20, 20)
    oled.show()
    time.sleep(0.5)
    
    oled.fill(0)         # Clear = text disappears
    oled.show()
    time.sleep(0.5)

# Two-screen slideshow:
while True:
    # Screen 1: Name
    oled.fill(0)
    oled.text("Hi, I'm Alex", 0, 20)
    oled.show()
    time.sleep(2)
    
    # Screen 2: Fun fact
    oled.fill(0)
    oled.text("I love robots!", 0, 20)
    oled.show()
    time.sleep(2)
```

Get creative! Ask your instructor for help if you get stuck!

---
## Vocabulary Recap

| Word | What It Means |
|------|---------------|
| **I2C** | A communication protocol that lets chips talk to each other using just 2 wires (SDA and SCL) |
| **SDA** | Serial Data line — carries the actual data in I2C communication |
| **SCL** | Serial Clock line — keeps the timing synchronized between devices |
| **Address** | A unique number each I2C device has so the Pico knows which device to talk to (OLED = 0x3C) |
| **OLED** | Organic Light-Emitting Diode — a display where each pixel makes its own light |
| **SSD1306** | The controller chip inside the OLED display module |
| **Library** | A file of pre-written code that adds new features (like `ssd1306.py` for the display) |
| **Pixel** | The smallest dot on a screen — the display is 128 x 64 pixels |

---
*Circuit Explorers — Module 12: Display It!*